## For downloading and calling the matlab function, run this cell. The code deletes the downloaded files automatically.

In [ ]:
import os
import sys
import tempfile
import requests
import subprocess
from urllib.parse import quote

USERNAME = "zahrabah@helsinki.fi"
APP_PASSWORD = "gy8XR-Akg7D-gMQdy-9Tb28-4wdQn"

BASE_URL_VEL = f"https://datacloud.helsinki.fi/remote.php/dav/files/{USERNAME}/totah-lab/projects/near_mistakes_dmitrii/raw_velocity_data/"
BASE_URL_BEH = f"https://datacloud.helsinki.fi/remote.php/dav/files/{USERNAME}/totah-lab/projects/near_mistakes_dmitrii/behavior_data/"

MATLAB = "/appl/manual_installations/software/matlab/r2024b/bin/matlab"


def download_file(base_url, fileName, savePath):
    url = base_url + quote(fileName)
    r = requests.get(url, auth=(USERNAME, APP_PASSWORD))
    r.raise_for_status()

    with open(savePath, "wb") as f:
        f.write(r.content)


ratId = sys.argv[1]
dateStr = sys.argv[2]
behFile = sys.argv[3]
nevFile = sys.argv[4]
ncsFile = sys.argv[5]

print("Processing:", ratId, dateStr)

with tempfile.TemporaryDirectory() as tmpdir:

    download_file(BASE_URL_BEH, behFile, os.path.join(tmpdir, behFile))
    download_file(BASE_URL_VEL, nevFile, os.path.join(tmpdir, nevFile))
    download_file(BASE_URL_VEL, ncsFile, os.path.join(tmpdir, ncsFile))

    subprocess.run([
        MATLAB,
        "-batch",
        f"rawrec2dataset('{ratId}', '{dateStr}', '{tmpdir}', '{behFile}', '{nevFile}', '{ncsFile}')"
    ], check=True)

## For just downloading the files from data cloud, run this cell:

In [ ]:
import os
import requests
from urllib.parse import quote

USERNAME = "zahrabah@helsinki.fi"
APP_PASSWORD = "gy8XR-Akg7D-gMQdy-9Tb28-4wdQn"

BASE_URL_VEL = f"https://datacloud.helsinki.fi/remote.php/dav/files/{USERNAME}/totah-lab/projects/near_mistakes_dmitrii/raw_velocity_data/"
BASE_URL_BEH = f"https://datacloud.helsinki.fi/remote.php/dav/files/{USERNAME}/totah-lab/projects/near_mistakes_dmitrii/behavior_data/"

JOBS_FILE = "jobs_rerun_manual.txt"
OUT_DIR = "/scratch/work/bahriz1/Thesis/rerun_downloads"

os.makedirs(OUT_DIR, exist_ok=True)


def download_file(base_url, file_name, save_path):
    url = base_url + quote(file_name)

    if os.path.exists(save_path):
        print("Already exists:", save_path)
        return

    r = requests.get(url, auth=(USERNAME, APP_PASSWORD), timeout=300)
    r.raise_for_status()

    with open(save_path, "wb") as f:
        f.write(r.content)

    print("Downloaded:", save_path)


with open(JOBS_FILE, "r") as f:
    for line in f:
        line = line.strip()

        if not line:
            continue

        ratId, dateStr, behFile, nevFile, ncsFile = line.split("|")

        session_dir = os.path.join(OUT_DIR, f"{ratId}_{dateStr}")
        os.makedirs(session_dir, exist_ok=True)

        print("Processing:", ratId, dateStr)

        download_file(BASE_URL_BEH, behFile, os.path.join(session_dir, behFile))
        download_file(BASE_URL_VEL, nevFile, os.path.join(session_dir, nevFile))
        download_file(BASE_URL_VEL, ncsFile, os.path.join(session_dir, ncsFile))

In [9]:
import os
import requests
from urllib.parse import quote

USERNAME = "zahrabah@helsinki.fi"
APP_PASSWORD = "gy8XR-Akg7D-gMQdy-9Tb28-4wdQn"

BASE_URL_VEL = f"https://datacloud.helsinki.fi/remote.php/dav/files/{USERNAME}/totah-lab/projects/near_mistakes_dmitrii/raw_velocity_data/"
BASE_URL_BEH = f"https://datacloud.helsinki.fi/remote.php/dav/files/{USERNAME}/totah-lab/projects/near_mistakes_dmitrii/behavior_data/"


OUT_DIR = "/scratch/work/bahriz1/Thesis/rerun_downloads"

os.makedirs(OUT_DIR, exist_ok=True)


def download_file(base_url, file_name, save_path):
    url = base_url + quote(file_name)

    if os.path.exists(save_path):
        print("Already exists:", save_path)
        return

    r = requests.get(url, auth=(USERNAME, APP_PASSWORD), timeout=900)
    r.raise_for_status()

    with open(save_path, "wb") as f:
        f.write(r.content)

    print("Downloaded:", save_path)


ratId = "10501"
dateStr ="041119"
behFile = "Rat 10501 Lev5_GNG2Stim 04-Nov-2019.mat"
nevFile = "Events_10501_041119.nev"
ncsFile = "CSC95_10501_041119.ncs"
    
session_dir = os.path.join(OUT_DIR, f"{ratId}_{dateStr}")
os.makedirs(session_dir, exist_ok=True)

print("Processing:", ratId, dateStr)

download_file(BASE_URL_BEH, behFile, os.path.join(session_dir, behFile))
download_file(BASE_URL_VEL, nevFile, os.path.join(session_dir, nevFile))
download_file(BASE_URL_VEL, ncsFile, os.path.join(session_dir, ncsFile))

Processing: 10501 041119
Downloaded: /scratch/work/bahriz1/Thesis/rerun_downloads/10501_041119/Rat 10501 Lev5_GNG2Stim 04-Nov-2019.mat
Downloaded: /scratch/work/bahriz1/Thesis/rerun_downloads/10501_041119/Events_10501_041119.nev
Downloaded: /scratch/work/bahriz1/Thesis/rerun_downloads/10501_041119/CSC95_10501_041119.ncs


In [10]:
import os
import subprocess

MATLAB = "/appl/manual_installations/software/matlab/r2024b/bin/matlab"
BASE_DIR = "/scratch/work/bahriz1/Thesis/rerun_downloads"

jobs = [
    "10501|041119|Rat 10501 Lev5_GNG2Stim 04-Nov-2019.mat|Events_10501_041119.nev|CSC95_10501_041119.ncs"
]

for line in jobs:

    ratId, dateStr, behFile, nevFile, ncsFile = line.split("|")

    tmpdir = os.path.join(BASE_DIR, f"{ratId}_{dateStr}")

    print("\nProcessing:", ratId, dateStr)

    for fileName in [behFile, nevFile, ncsFile]:

        fullpath = os.path.join(tmpdir, fileName)

        print(fullpath)

        if not os.path.exists(fullpath):
            raise FileNotFoundError(fullpath)

    subprocess.run([
        MATLAB,
        "-batch",
        f"rawrec2dataset('{ratId}', '{dateStr}', '{tmpdir}', '{behFile}', '{nevFile}', '{ncsFile}')"
    ], check=True)


Processing: 10501 041119
/scratch/work/bahriz1/Thesis/rerun_downloads/10501_041119/Rat 10501 Lev5_GNG2Stim 04-Nov-2019.mat
/scratch/work/bahriz1/Thesis/rerun_downloads/10501_041119/Events_10501_041119.nev
/scratch/work/bahriz1/Thesis/rerun_downloads/10501_041119/CSC95_10501_041119.ncs


In [5]:
import requests

r = requests.get(
    BASE_URL_BEH,
    auth=(USERNAME, APP_PASSWORD)
)

print(r.status_code)
print(r.text[:500])

200



In [ ]:
r = requests.get(url, auth=(USERNAME, APP_PASSWORD), timeout=900)